In [3]:
# !pip install tensorflow
import tensorflow as tf
from tensorflow.keras import layers, models

def build_cnn_autoencoder(input_shape=(1024, 64, 3)):
    """
    정상 3상 전류 STFT 이미지를 학습하여 고장을 탐지하기 위한 CNN 오토인코더
    """
    # ==========================================
    # 1. 인코더 (Encoder) Z: 이미지의 핵심 특징만 압축 추출
    # ==========================================
    inputs = layers.Input(shape=input_shape)
    
    # 특징을 찾고(Conv2D) -> 크기를 절반으로 줄임(MaxPooling2D)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2), padding='same')(x) # (512, 32, 32)
    
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x) # (256, 16, 16)
    
    x = layers.Conv2D(8, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x) # (128, 8, 8) 
    
    # 완전 연결층(Dense)으로 극한의 압축 (공간 정보 파괴 후 특징만 추출)
    x = layers.Flatten()(x) # 128 * 8 * 8 = 8192
    encoded = layers.Dense(128, activation='relu')(x)
    # ↑ 진정한 1차원 병목 지점 (128개의 벡터)

    # ==========================================
    # 2. 디코더 (Decoder) : 압축된 특징을 바탕으로 원본 이미지 복원
    # ==========================================
    x = layers.Dense(8192, activation='relu')(encoded)
    x = layers.Reshape((128, 8, 8))(x)
    
    x = layers.Conv2D(8, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x) # 크기를 다시 2배로 키움 (256, 16, 8)
    
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x) # (512, 32, 16)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x) # (1024, 64, 32)
    
    # 마지막 복원층: 채널을 원본과 똑같이 3채널(R,G,B)로 맞추고, 
    # 데이터가 0~1 사이 정규화 값이므로 sigmoid 활성화 함수 사용
    decoded = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

    # ==========================================
    # 3. 전체 모델 조립 및 컴파일
    # ==========================================
    autoencoder = models.Model(inputs, decoded)
    
    # 원본 이미지와 복원된 이미지 간의 픽셀 오차(Mean Squared Error)를 최소화하도록 학습
    autoencoder.compile(optimizer='adam', loss='mse')
    
    return autoencoder

# 모델 생성 및 구조 확인
autoencoder = build_cnn_autoencoder()


In [4]:
import numpy as np
import glob

# 1. 학습 데이터(.npy) 로드
input_files = glob.glob('normal_image/*.npy')
X_train_list = []

for input_file in input_files:
    img = np.load(input_file)

    img_cropped = img[:1024, :, :]
    X_train_list.append(img_cropped)
X_train = np.array(X_train_list)
# print(X_train)
print(f"데이터 로드 완료 / shape: {X_train.shape}")

# 2. 모델 학습
history = autoencoder.fit(
    x=X_train, 
    y=X_train,           # 정답은 입력값 자기 자신
    epochs=20,           # 학습 반복 횟수
    batch_size=8,        # 한 번에 투입할 이미지 장 수
    validation_split=0.2 # 검증용 모의고사 데이터 비율 (20%)
)
print("학습 완료")


# 3. 학습된 정상 데이터들의 복원 오차 확인
reconstructed_images = autoencoder.predict(X_train)
mse = tf.keras.losses.mse(X_train, reconstructed_images)
reconstruction_errors = np.max(mse, axis=(1,2)) # 픽셀 전체 평균이 아닌, '가장 크게 틀린(Max) 픽셀' 기준! 
max_normal_error = np.max(reconstruction_errors)


데이터 로드 완료 / shape: (1918, 1024, 64, 3)
Epoch 1/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 20s 93ms/step - loss: 0.0060 - val_loss: 0.0015
Epoch 2/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 17s 90ms/step - loss: 0.0013 - val_loss: 0.0013
Epoch 3/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 18s 96ms/step - loss: 0.0012 - val_loss: 0.0012
Epoch 4/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 19s 97ms/step - loss: 0.0012 - val_loss: 0.0012
Epoch 5/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 19s 97ms/step - loss: 0.0012 - val_loss: 0.0012
Epoch 6/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 16s 85ms/step - loss: 0.0012 - val_loss: 0.0012
Epoch 7/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 17s 88ms/step - loss: 0.0012 - val_loss: 0.0012
Epoch 8/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 18s 92ms/step - loss: 0.0012 - val_loss: 0.0012
Epoch 9/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 17s 89ms/step - loss: 0.0011 - val_loss: 0.0011
Epoch 10/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 16s 85ms/step - loss: 0.0011 - val_loss: 0.0011
Epoch 11/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 16s 83ms/step - loss: 0.0011 - val_loss

In [5]:
# 2. 통계치 계산
mean_error = np.mean(reconstruction_errors)
std_error = np.std(reconstruction_errors)
max_error = np.max(reconstruction_errors)
# 3. 임계값(Threshold) 산출
threshold_3sigma = mean_error + (3 * std_error)
threshold_max = max_error * 1.05  # 최대값의 5% 마진
print("=== 정상 데이터 오차율 통계 ===")
print(f"평균 오차율: {mean_error:.6f}")
print(f"표준 편차: {std_error:.6f}")
print(f"최대 오차율: {max_error:.6f}")
print("\n=== 추천 임계값(Threshold) ===")
print(f"추천 1 (3-Sigma 정석) : {threshold_3sigma:.6f}")
print(f"추천 2 (Max + 여유분) : {threshold_max:.6f}")

=== 정상 데이터 오차율 통계 ===
평균 오차율: 0.035743
표준 편차: 0.008886
최대 오차율: 0.085379

=== 추천 임계값(Threshold) ===
추천 1 (3-Sigma 정석) : 0.062400
추천 2 (Max + 여유분) : 0.089647


In [6]:
import numpy as np
import glob

abnormal_files = glob.glob('abnormal_image/*.npy')
abnormal_list = []

for abnormal_file in abnormal_files:
    img = np.load(abnormal_file)

    img_cropped = img[:1024, :, :]
    abnormal_list.append(img_cropped)
    
X_abnormal = np.array(abnormal_list)

abnormal_images = autoencoder.predict(X_abnormal)
abnormal_mse = tf.keras.losses.mse(X_abnormal, abnormal_images)
abnormal_errors = np.max(abnormal_mse, axis=(1,2)) # 픽셀 전체 평균이 아닌, '가장 크게 틀린(Max) 픽셀' 기준! 
print(abnormal_errors)
print(np.mean(abnormal_errors))

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step
[0.04434244 0.03629419 0.04380872 0.0382663  0.03328439 0.04545565
 0.0669322  0.04825331 0.03828265 0.04843308 0.04457855 0.03095677
 0.04217865 0.06850103 0.03216682 0.03225683 0.05012918 0.03812774
 0.03738144 0.04176867 0.03045159 0.02878787 0.03780295 0.04938899
 0.04044842 0.03256716 0.02635364 0.04053422 0.02618966 0.03088686
 0.02971103 0.03559318 0.0458896  0.03773862 0.03336302 0.04641992
 0.03203614 0.03246943 0.03933698 0.03751968 0.0248317  0.03726439
 0.03438092 0.04382855 0.03275025 0.03949416 0.03594816 0.03755316
 0.0400028  0.03742816 0.04462263 0.03490689 0.02920507 0.03601145
 0.03762453 0.03198553 0.03221959 0.0305385  0.02814768 0.03894275
 0.03972456 0.02877823 0.04044643 0.04155355 0.05969722 0.04578806
 0.03991034 0.02950264 0.03341293 0.02795602 0.03131595 0.03920113
 0.05819724 0.06762723 0.03060534 0.04020054 0.0335743  0.0337611
 0.07036891 0.06584083 0.03400209 0.04436148 0.04087628 0.03353165
 0.03379242 0.03047878